# ClinIQ — Stage C: Teacher SFT Training

Fine-tunes Qwen3.5-9B on clinical triage data using Unsloth + LoRA.

In [ ]:
# @title ## 🔐 Enter Your HF Token
HF_TOKEN = ""  # @param {type:"string"}
import os
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# @title ## 📦 Install Dependencies
!pip install -q unsloth trl transformers torch datasets accelerate

In [ ]:
# @title ## 📥 Clone Repo & Prepare Data
!git clone https://github.com/Zolisa/Knowldge-Distillation103.git /content/cliniq
%cd /content/cliniq
!pip install -q -r requirements.txt
!python data/fetch_regulations.py
!python data/prepare.py

In [ ]:
# @title ## 🚀 Run Teacher SFT Training
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from data.dataset import load_sft_dataset

# Model config
MODEL_NAME = "Qwen/Qwen3.5-9B"
MAX_SEQ_LENGTH = 2048
LORA_R = 16

# Load model with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
)

# Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
)

# Load dataset
train_dataset = load_sft_dataset("data/processed/train.jsonl", tokenizer, MAX_SEQ_LENGTH)
eval_dataset = load_sft_dataset("data/processed/eval.jsonl", tokenizer, MAX_SEQ_LENGTH)

# Training args
training_args = TrainingArguments(
    output_dir="./outputs/stage_c",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    warmup_ratio=0.1,
)

# Train
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    tokenizer=tokenizer,
    packing=False,
)

trainer.train()

In [ ]:
# @title ## 💾 Save Model to HF
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)

# Save adapter
model.save_pretrained("cliniq-teacher")
tokenizer.save_pretrained("cliniq-teacher")

# Push to HF
api = HfApi()
api.upload_folder(
    folder_path="cliniq-teacher",
    repo_id="Zolisa/cliniq-teacher",
    repo_type="model",
)